# Phase 17 — Le faux témoignage

## Objectifs

- Produire un faux témoignage que le Conseil ne saura pas distinguer des vrais, sans modifier la
  moindre valeur interne du modèle emprunté — seul levier autorisé : la façon dont le modèle choisit
  chaque mot au moment de l'écrire (température, échantillonnage).
- Montrer les deux échecs (texte propre et répétitif ; texte incohérent qui invente des mots),
  chercher méthodiquement le point utilisable entre les deux.
- Test en aveugle : mélanger faux et vrais relevés, trier sans connaître les étiquettes, rendre le
  résultat tel quel.


## Changement de modèle emprunté, et pourquoi

Le modèle emprunté en phase 14 (`distilbert-base-uncased`) est un **encodeur** : il n'a ni tête de
génération, ni notion de « mot suivant ». Cette phase exige justement de choisir un mot après l'autre
— une tâche pour laquelle un encodeur n'est pas outillé, quel que soit le réglage de décodage. On
emprunte donc un second modèle, dans le même esprit (petit, libre, tournant sur CPU) mais de la bonne
famille : `distilgpt2`, un modèle décodeur autorégressif (82 M de paramètres).

## 1. Imports

In [1]:
from pathlib import Path
import csv
import hashlib
import random
import re

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


## 2. Configuration

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")
NOM_MODELE_GENERATEUR = "distilgpt2"

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
PHASE17_DIR = OUTPUT_DIR / "phase_17_faux_temoignage"
PHASE17_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = DATA_DIR / "releves_klaxo3.csv"

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

LONGUEUR_MAX_GENERATION = 40  # jetons, coherent avec la longueur des vrais relenves (phase 6 : 55 max, 14 median)


## 3. Le modèle emprunté et l'étalon de style (vrais relevés)

In [3]:
tokenizer_gen = AutoTokenizer.from_pretrained(NOM_MODELE_GENERATEUR)
tokenizer_gen.pad_token = tokenizer_gen.eos_token
modele_gen = AutoModelForCausalLM.from_pretrained(NOM_MODELE_GENERATEUR)
modele_gen.eval()

def empreinte_poids(modele):
    with torch.no_grad():
        morceaux = [p.detach().numpy().tobytes() for p in modele.parameters()]
    return hashlib.sha256(b"".join(morceaux)).hexdigest()

empreinte_avant = empreinte_poids(modele_gen)
print(f"Empreinte des poids avant génération : {empreinte_avant[:16]}...")

lignes_valides = []
with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)
df = pd.DataFrame(lignes_valides, columns=COLUMNS)
df["comments_clean"] = df["comments"].fillna("").astype(str).str.strip()

def tokenizer_simple(texte):
    return re.findall(r"[a-z0-9]+", str(texte).lower())

echantillon_reel = df.loc[
    df["comments_clean"].str.split().str.len().between(8, 20)
    & df["comments_clean"].ne("")
].sample(n=200, random_state=SEED)

print(f"Étalon de style : {len(echantillon_reel)} vrais relevés (8-20 mots).")
for texte in echantillon_reel["comments_clean"].head(3):
    print(" -", texte)


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Empreinte des poids avant génération : a6b9f68a1f37425e...


Étalon de style : 200 vrais relevés (8-20 mots).
 - First time experience they said nasa was hiding a secret mission from the people.
 - A Silent orange light flew over me and my family
 - Silver object seen hovering over marsh in West Dennis during daytime on 9-11-12


## 4. Générer, à un seul réglage fixe, pour voir les deux échecs

Un premier essai avec une amorce neutre (« I was ») produisait un texte fluide mais générique — du
sport, un professeur, un entraîneur de football — rien à voir avec un témoignage d'OVNI :
`distilgpt2` est un modèle de langue généraliste, jamais vu la transmission, et une amorce neutre ne
lui donne aucune raison de rester sur le sujet. La règle absolue de cette phase interdit de toucher
aux poids (pas de fine-tuning), mais rien n'interdit de choisir un **contexte de départ** dans le
registre voulu : ce n'est pas une valeur interne du modèle, c'est une entrée, au même titre qu'une
question l'est pour le système de la phase 15. On amorce donc chaque génération avec le tout début
d'un vrai relevé (3 à 5 premiers mots), tiré au hasard dans un petit lot d'amorces réelles — la suite
est entièrement écrite par le modèle, seule température (et l'échantillonnage top-p) change d'un
essai à l'autre. Aucun poids du modèle n'est touché.

In [4]:
AMORCES_REELLES = (
    echantillon_reel["comments_clean"]
    .apply(lambda t: " ".join(t.split()[:4]))
    .drop_duplicates()
    .sample(n=20, random_state=SEED)
    .tolist()
)
print("Exemples d'amorces réelles utilisées :", AMORCES_REELLES[:5])

def generer(temperature, top_p=0.95, top_k=50, max_new_tokens=LONGUEUR_MAX_GENERATION, seed_local=SEED):
    torch.manual_seed(seed_local)
    amorce = AMORCES_REELLES[seed_local % len(AMORCES_REELLES)]
    entree = tokenizer_gen(amorce, return_tensors="pt")
    with torch.no_grad():
        sortie = modele_gen.generate(
            **entree,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-4),
            top_p=top_p,
            top_k=top_k,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer_gen.eos_token_id,
        )
    return tokenizer_gen.decode(sortie[0], skip_special_tokens=True)

texte_trop_froid = generer(temperature=0.05)
texte_trop_chaud = generer(temperature=2.0)

print("Échec 1 — température quasi nulle (répétitif, se reconnaît en 2 secondes) :")
print(" ", repr(texte_trop_froid))
print()
print("Échec 2 — température très haute (part n'importe où) :")
print(" ", repr(texte_trop_chaud))


Exemples d'amorces réelles utilisées : ['I JUST HAPPENED TO', 'Small blue light traveling', 'flahing &quot;star&quot; moving around', 'V-shaped UFO Appears Across', 'Triangular and low-flying aircraft']


Échec 1 — température quasi nulle (répétitif, se reconnaît en 2 secondes) :
  'flahing &quot;star&quot; moving around, and moving around, and moving around, and moving around, and moving around, and moving around, and moving around, and moving around, and moving around, and moving around, and moving around'

Échec 2 — température très haute (part n'importe où) :
  'flahing &quot;star&quot; moving around 1 hour ago'


## 5. Recherche méthodique du point utilisable

Deux mesures objectives, pas un réglage à l'oreille : le **taux de répétition** (part des trigrammes
de mots dupliqués dans le texte généré — un texte qui boucle en a beaucoup) et sa comparaison au taux
de répétition mesuré sur les **vrais** relevés (l'étalon, section 3). On balaie une grille de
températures et on retient la plus basse dont le taux de répétition redescend au niveau des vrais
relevés — la plus basse, parce qu'une température inutilement haute n'achète rien et ne fait
qu'augmenter le risque d'incohérence.

In [5]:
def taux_repetition(texte):
    mots = tokenizer_simple(texte)
    if len(mots) < 3:
        return 0.0
    trigrammes = [tuple(mots[i:i+3]) for i in range(len(mots) - 2)]
    if not trigrammes:
        return 0.0
    return 1 - len(set(trigrammes)) / len(trigrammes)

taux_repetition_reel_moyen = np.mean([taux_repetition(t) for t in echantillon_reel["comments_clean"]])
print(f"Taux de répétition moyen, vrais relevés (étalon) : {taux_repetition_reel_moyen:.4f}")

GRILLE_TEMPERATURES = [0.05, 0.3, 0.5, 0.7, 0.9, 1.1, 1.4, 1.8]
resultats_grille = []
for temperature in GRILLE_TEMPERATURES:
    textes = [generer(temperature, seed_local=SEED + i) for i in range(5)]
    taux = np.mean([taux_repetition(t) for t in textes])
    resultats_grille.append({"temperature": temperature, "taux_repetition_moyen": taux, "exemple": textes[0]})
    print(f"T={temperature:>4.2f} | taux répétition={taux:.4f} | exemple : {textes[0][:90]!r}")

tableau_grille = pd.DataFrame(resultats_grille)


Taux de répétition moyen, vrais relevés (étalon) : 0.0003


T=0.05 | taux répétition=0.1588 | exemple : 'flahing &quot;star&quot; moving around, and moving around, and moving around, and moving a'


T=0.30 | taux répétition=0.1588 | exemple : 'flahing &quot;star&quot; moving around, and moving around, and moving around, and moving a'


T=0.50 | taux répétition=0.4846 | exemple : 'flahing &quot;star&quot; moving around, and moving around, and moving around, and moving a'


T=0.70 | taux répétition=0.3095 | exemple : 'flahing &quot;star&quot; moving around, and moving around, and moving around, and moving a'


T=0.90 | taux répétition=0.2731 | exemple : 'flahing &quot;star&quot; moving around, and moving around, and moving around, and moving a'


T=1.10 | taux répétition=0.0000 | exemple : 'flahing &quot;star&quot; moving around with them!'


T=1.40 | taux répétition=0.0111 | exemple : 'flahing &quot;star&quot; moving around with them!'


T=1.80 | taux répétition=0.0000 | exemple : 'flahing &quot;star&quot; moving around 1 hour ago'


In [6]:
candidats_valables = tableau_grille.loc[tableau_grille["taux_repetition_moyen"] <= taux_repetition_reel_moyen * 1.5]
TEMPERATURE_RETENUE = float(candidats_valables["temperature"].min()) if len(candidats_valables) else float(tableau_grille["temperature"].median())

print(f"Température retenue : {TEMPERATURE_RETENUE} "
      f"(la plus basse dont le taux de répétition s'approche de l'étalon réel {taux_repetition_reel_moyen:.4f})")


Température retenue : 1.1 (la plus basse dont le taux de répétition s'approche de l'étalon réel 0.0003)


## 6. Cinq faux témoignages, au réglage retenu

In [7]:
faux_temoignages = [generer(TEMPERATURE_RETENUE, seed_local=SEED + 100 + i) for i in range(5)]
for i, t in enumerate(faux_temoignages, start=1):
    print(f"Faux {i} : {t}")


Faux 1 : flahing &quot;star&quot; moving around.
The original story first appeared at the SEGA Mobile booth in Tokyo on January 19, 2016 and the story has since been updated to include a new gameplay mechanic, and a new feature!

Faux 2 : V-shaped UFO Appears Across the West


The Flying and Flying-Stereogies
Aerial view from the top of the sky of Tauron
A UFO is seen in the sky of Mount Vesuv
Faux 3 : Triangular and low-flying aircraft were seen, with the first aircraft being fitted to the air.
Faux 4 : Red ball of light and light. A round, ball of light as it moves from its surface. The light and light travel back towards the ball's surface and forth as it moves. This is the position of the ball
Faux 5 : Portland OR May &#3905 - May &#4018 - May &#4028 - May &#4027 - May &#4026 - May &#4025 - May &#4024 - May &#


## 7. Preuve que le modèle n'a pas bougé

In [8]:
empreinte_apres = empreinte_poids(modele_gen)
print(f"Empreinte avant : {empreinte_avant[:16]}...")
print(f"Empreinte après : {empreinte_apres[:16]}...")
print(f"Poids strictement inchangés entre le premier essai et le dernier : {empreinte_avant == empreinte_apres}")
assert empreinte_avant == empreinte_apres


Empreinte avant : a6b9f68a1f37425e...
Empreinte après : a6b9f68a1f37425e...
Poids strictement inchangés entre le premier essai et le dernier : True


## 8. Test en aveugle : mélanger, exporter sans étiquette

Cinq faux (section 6) et cinq vrais relevés de longueur comparable, mélangés avec une permutation
tirée au hasard. Le fichier livré pour le tri **ne contient pas** l'étiquette vrai/faux — seulement un
identifiant anonyme et le texte. La clé de réponse est sauvegardée à part, non affichée dans cette
cellule.

In [9]:
vrais_pour_test = df.loc[
    df["comments_clean"].str.split().str.len().between(8, 15) & df["comments_clean"].ne("")
].sample(n=5, random_state=SEED + 7)["comments_clean"].tolist()

candidats = [(t, "faux") for t in faux_temoignages] + [(t, "vrai") for t in vrais_pour_test]
random.Random(SEED).shuffle(candidats)

table_aveugle = pd.DataFrame({
    "id_anonyme": [f"R{i+1}" for i in range(len(candidats))],
    "texte": [c[0] for c in candidats],
})
cle_reponse = pd.DataFrame({
    "id_anonyme": [f"R{i+1}" for i in range(len(candidats))],
    "etiquette_reelle": [c[1] for c in candidats],
})

table_aveugle.to_csv(PHASE17_DIR / "test_aveugle_sans_etiquette.csv", index=False)
cle_reponse.to_csv(PHASE17_DIR / "test_aveugle_cle_reponse.csv", index=False)

print("Fichier remis pour le tri (sans étiquette) :")
table_aveugle


Fichier remis pour le tri (sans étiquette) :


,id_anonyme,texte
0,R1,It was dusk it all lasted about 20 min the obj...
1,R2,"Red ball of light and light. A round, ball of ..."
2,R3,"Triangular and low-flying aircraft were seen, ..."
3,R4,Saucer with lots of lights flying at about 3&#...
4,R5,Mysterious flashing stars appear and disapear ...
5,R6,9 CROSS-SHAPED GLOWING&#44 RED/ORANGE/YELLOW L...
6,R7,Rectangular object in the sky above Orlando&#...
7,R8,Portland OR May &#3905 - May &#4018 - May &#40...
8,R9,flahing &quot;star&quot; moving around.\nThe o...
9,R10,V-shaped UFO Appears Across the West\n\n\nThe ...


## 9. Résultat du tri en aveugle

Tri réalisé par l'auteur du rapport, verdicts écrits pour les 10 relevés **avant** toute consultation
de `test_aveugle_cle_reponse.csv` — limite reconnue : ce n'est pas un tiers extérieur au projet, la
procédure est donc un test en aveugle affaibli, pas un vrai test à l'insu d'un juge indépendant.

| id | Verdict avant ouverture de la clé | Indice qui a guidé le verdict | Étiquette réelle |
|---|---|---|---|
| R1 | Vrai | Style factuel terse, typique d'un relevé court | Vrai |
| R2 | Faux | Répétition circulaire (« light and light », « as it moves... as it moves ») et fin qui ne referme pas la phrase | Faux |
| R3 | Faux | Fin incohérente : « fitted to the air » ne veut rien dire | Faux |
| R4 | Vrai | Style terse plausible, cohérent de bout en bout | Vrai |
| R5 | Vrai | Court, plausible, faute de frappe (« disapear ») cohérente avec un vrai témoignage | Vrai |
| R6 | Vrai | Registre tout en majuscules, typique d'un titre de relevé réel | Vrai |
| R7 | Vrai | Simple, plausible, format standard | Vrai |
| R8 | Faux | Dérive vers une suite de dates cassées et répétées, non-sens | Faux |
| R9 | Faux | Dérive complète hors sujet (un stand SEGA Mobile à Tokyo) | Faux |
| R10 | Faux | Incohérent, invente un mot (« Flying-Stereogies ») | Faux |

**10 / 10 corrects.** Le test en aveugle échoue à démontrer que les faux sont indiscernables — c'est
le contraire : chacun des 5 faux porte une trace reconnaissable, presque toujours la même famille de
symptôme que les deux échecs de la section 4, simplement moins prononcée à T = 1,1 qu'aux extrêmes :
une dérive de sujet en fin de génération, ou une incohérence locale (un mot inventé, une phrase qui ne
se referme pas). La grille de température (section 5) optimisait un seul critère mesurable — le taux
de répétition — qui distingue bien l'échec 1 (la boucle) mais ne dit rien sur l'échec 2 (la dérive
sémantique et l'invention) : `distilgpt2`, généraliste et non spécialisé sur ce corpus, dérive presque
toujours au bout d'une vingtaine de jetons, quelle que soit la température, dès qu'il quitte le
sillage de l'amorce réelle. Un texte plus court (moins de jetons générés, donc moins de distance
parcourue depuis l'amorce) ou un modèle plus gros auraient probablement rendu le tri plus difficile —
ce n'est pas testé ici, faute de budget de calcul.

## 10. Export

In [10]:
tableau_grille.to_csv(PHASE17_DIR / "grille_temperatures.csv", index=False)

resume_phase17 = pd.DataFrame([{
    "modele_generateur": NOM_MODELE_GENERATEUR,
    "temperature_retenue": TEMPERATURE_RETENUE,
    "taux_repetition_reel_etalon": taux_repetition_reel_moyen,
    "taux_repetition_a_temp_retenue": float(tableau_grille.loc[tableau_grille["temperature"] == TEMPERATURE_RETENUE, "taux_repetition_moyen"].iloc[0]),
    "poids_inchanges": bool(empreinte_avant == empreinte_apres),
    "nombre_faux_generes": len(faux_temoignages),
}])
resume_phase17.to_csv(PHASE17_DIR / "resume_phase17.csv", index=False)
resume_phase17


,modele_generateur,temperature_retenue,taux_repetition_reel_etalon,taux_repetition_a_temp_retenue,poids_inchanges,nombre_faux_generes
0,distilgpt2,1.1,0.00025,0.0,True,5
